# RQ5, Part 2: Real Bootstrap Confidence Interval (Four-Project Version)

**This notebook replaces the previous version of `RQ5_2_Bootstrap_CI.ipynb`,** which grouped Camel and Hadoop as a single combined project (via `rq1_refined_with_real_issue_type.csv`), predating the RQ1 correction that established four independently tested projects. This version resamples and recomputes the effect size for each of the four projects separately, matching `RQ5_1_Point_Estimate_Synthesis.ipynb` and the final report exactly.

Also includes the censoring-aware RQ4 sensitivity variant (observed cases only, N=61) alongside the primary full-sample RQ4 model (N=113), since the final report discloses both.

**Requires the same real data as Part 1.**

In [1]:
# --- Setup: make this notebook runnable standalone in Colab or locally ---
import os

REPO_URL = "https://github.com/daljeetkaurJohar/qm640-governance-analytics.git"
REPO_DIR = "qm640-governance-analytics"

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if in_colab():
    # Colab starts in /content with no repo checked out -- clone it once,
    # then cd into it so the relative "data/cleaned/..." paths below resolve.
    if not os.path.isdir(f"/content/{REPO_DIR}"):
        get_ipython().system(f"git clone --depth 1 {REPO_URL} /content/{REPO_DIR}")
    os.chdir(f"/content/{REPO_DIR}")
else:
    # Running locally: assume this notebook is being run from notebooks/ inside
    # the cloned repo (its normal location) and step up to the repo root, where
    # the "data/cleaned/..." paths below expect to be run from.
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")

print("Working directory:", os.getcwd())
assert os.path.isfile("data/cleaned/camel_real_mined_dataset.csv"), (
    "camel_real_mined_dataset.csv not found -- check that the repo cloned/changed "
    "directory correctly above, or that you are running from the repo root."
)


Cloning into '/content/qm640-governance-analytics'...
remote: Enumerating objects: 107, done.
remote: Counting objects: 100% (107/107), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 107 (delta 22), reused 59 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (107/107), 7.20 MiB | 9.89 MiB/s, done.
Resolving deltas: 100% (22/22), done.
Working directory: /content/qm640-governance-analytics


In [2]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

np.random.seed(42)
N_BOOT = 1000


## Caveat: comparing f² across model families

The RQ1 QA-domain effect sizes come from **logistic regression** models (binary defect-prone outcome) via `1 - (llf / llnull)` pseudo-R², while the RQ4 audit-domain effect size comes from an **OLS** model (continuous remediation-days outcome, N=113) via ordinary R². Cohen's f² is computed the same way in both cases (`(R2_full - R2_reduced) / (1 - R2_full)`), but the underlying R²/pseudo-R² quantities are not measuring the same thing: OLS R² is the share of outcome variance explained; McFadden's pseudo-R² does not have that direct variance-explained interpretation and is known to run numerically smaller than OLS R² for comparable model fit.

This means the gap and its bootstrap distribution should be read as a comparison of two Cohen's-f²-scaled quantities, not as two directly equivalent "percent of variance explained" figures. The report states this comparability limitation explicitly rather than implying the two f² values are on an identical footing.

In [3]:
def rq1_f2(df, target):
    df = df.copy()
    df["era_binary"] = (df["era"] == "ai_era").astype(int)
    try:
        full = smf.logit(
            f"{target} ~ (loc + cyclomatic_complexity + num_functions + num_files_changed) * era_binary",
            data=df).fit(disp=0)
        reduced = smf.logit(
            f"{target} ~ loc + cyclomatic_complexity + num_functions + num_files_changed + era_binary",
            data=df).fit(disp=0)
        r2f = 1 - (full.llf / full.llnull)
        r2r = 1 - (reduced.llf / reduced.llnull)
        return (r2f - r2r) / (1 - r2f)
    except Exception:
        return np.nan

def rq4_f2(df, censoring_aware=False):
    if censoring_aware:
        df = df[df["event_observed"] == 1].copy()
        if len(df) < 15 or df["weakness_group"].nunique() < 2:
            return np.nan
        df["disclosure_year_c"] = df["disclosure_year"] - df["disclosure_year"].mean()
        yr_term = "disclosure_year_c"
    else:
        yr_term = "disclosure_year"
    try:
        full = smf.ols(f"remediation_days ~ C(weakness_group) + C(industry_group) + {yr_term}", data=df).fit()
        reduced = smf.ols("remediation_days ~ C(weakness_group)", data=df).fit()
        return (full.rsquared - reduced.rsquared) / (1 - full.rsquared)
    except Exception:
        return np.nan

def consolidate_weakness(cat):
    cat = str(cat)
    if "Revenue" in cat: return "Revenue Recognition"
    if "ITGC" in cat: return "ITGC"
    if "Complex" in cat or "Warrant" in cat or "Instrument" in cat: return "Complex Transactions/Instruments"
    if "Control Environment" in cat or "Staffing" in cat or "Risk Assessment" in cat or "Segregation" in cat: return "Control Environment/Staffing"
    return "Other"

def consolidate_industry(ind):
    ind = str(ind)
    if "Technology" in ind: return "Technology"
    if "Biotech" in ind or "Healthcare" in ind: return "Biotech/Healthcare"
    if "Manufacturing" in ind or "Industrial" in ind or "Aerospace" in ind or "Mining" in ind: return "Manufacturing/Industrial"
    if "SPAC" in ind: return "SPAC"
    if "Energy" in ind: return "Energy"
    if "Financial" in ind or "Insurance" in ind or "Real Estate" in ind: return "Financial/Real Estate"
    return "Media/Consumer/Other"


In [4]:
camel = pd.read_csv("data/cleaned/camel_real_mined_dataset.csv")
hadoop = pd.read_csv("data/cleaned/hadoop_real_mined_dataset.csv")
kafka = pd.read_csv("data/cleaned/kafka_real_mined_dataset.csv")
tika = pd.read_csv("data/cleaned/tika_real_mined_dataset.csv")

rq4 = pd.read_csv("data/cleaned/rq4_real_sec_edgar_dataset_FINAL.csv", parse_dates=["disclosure_date", "remediation_date"])
rq4["weakness_group"] = rq4["weakness_category"].apply(consolidate_weakness)
rq4["industry_group"] = rq4["industry"].apply(consolidate_industry)
rq4["disclosure_year"] = rq4["disclosure_date"].dt.year


## Bootstrap: 1,000 resamples, four independent RQ1 projects

In [5]:
qa_avgs, gaps_old, gaps_corr = [], [], []

for b in range(N_BOOT):
    c_s = camel.sample(frac=1, replace=True, random_state=b)
    h_s = hadoop.sample(frac=1, replace=True, random_state=b + 1)
    k_s = kafka.sample(frac=1, replace=True, random_state=b + 2)
    t_s = tika.sample(frac=1, replace=True, random_state=b + 3)
    r4_s = rq4.sample(frac=1, replace=True, random_state=b + 4)

    qa_avg = np.nanmean([
        rq1_f2(c_s, "defect_prone_strict"),
        rq1_f2(h_s, "defect_prone_strict"),
        rq1_f2(k_s, "defect_prone"),
        rq1_f2(t_s, "defect_prone"),
    ])
    if np.isnan(qa_avg):
        continue
    qa_avgs.append(qa_avg)

    gaps_old.append(abs(rq4_f2(r4_s, censoring_aware=False) - qa_avg))
    corr = rq4_f2(r4_s, censoring_aware=True)
    if not np.isnan(corr):
        gaps_corr.append(abs(corr - qa_avg))

    if (b + 1) % 200 == 0:
        print(f"  {b + 1}/{N_BOOT} bootstrap resamples complete")

qa_avgs = np.array(qa_avgs)
gaps_old = np.array(gaps_old)
gaps_corr = np.array(gaps_corr)

print(f"\nReal bootstrap resamples completed: {len(qa_avgs)}")


  200/1000 bootstrap resamples complete
  400/1000 bootstrap resamples complete
  600/1000 bootstrap resamples complete
  800/1000 bootstrap resamples complete
  1000/1000 bootstrap resamples complete

Real bootstrap resamples completed: 1000


## Results: original RQ4 model (all N=113)

In [6]:
print(f"QA domain f2: mean={qa_avgs.mean():.4f}, 95% CI=[{np.percentile(qa_avgs,2.5):.4f}, {np.percentile(qa_avgs,97.5):.4f}]")
print(f"Gap |audit - QA| (old RQ4): mean={gaps_old.mean():.4f}, 95% CI=[{np.percentile(gaps_old,2.5):.4f}, {np.percentile(gaps_old,97.5):.4f}]")
print(f"Proportion of bootstrap gaps below 0.10 threshold: {(gaps_old < 0.10).mean()*100:.1f}%")


QA domain f2: mean=0.0128, 95% CI=[0.0056, 0.0224]
Gap |audit - QA| (old RQ4): mean=0.1207, 95% CI=[0.0265, 0.2712]
Proportion of bootstrap gaps below 0.10 threshold: 43.5%


## Results: censoring-aware RQ4 sensitivity model (N=61 observed cases only)

In [7]:
print(f"Gap |audit - QA| (censoring-aware RQ4): mean={gaps_corr.mean():.4f}, 95% CI=[{np.percentile(gaps_corr,2.5):.4f}, {np.percentile(gaps_corr,97.5):.4f}]")
print(f"Proportion of bootstrap gaps below 0.10 threshold: {(gaps_corr < 0.10).mean()*100:.1f}%")


Gap |audit - QA| (censoring-aware RQ4): mean=0.3625, 95% CI=[0.0659, 0.9953]
Proportion of bootstrap gaps below 0.10 threshold: 7.6%
